SETUP

In [1]:
import pandas as pd

run_id_main = 48768
run_id_comp = 46654
file_main = f"run/{run_id_main}/results_{run_id_main}.tsv"
file_comp = f"run/{run_id_comp}/results_{run_id_comp}.tsv"

In [2]:
# main dataframe
df = pd.read_csv(file_main, sep='\t', header=0)
df.head()

,ID,max_iters,hash
0,00063d88244921d6ec46aeab6866a8e2,4,dde758fade034d78
1,00063d88244921d6ec46aeab6866a8e2,12,3957e818cea23618
2,00072cf107ae1349c8c59a15c5ce4af1,4,0f413337838a07ef
3,00072cf107ae1349c8c59a15c5ce4af1,12,0f413337838a07ef
4,00076733bdbce94d7e44eca84f1425f0,4,1091ad934880f8f2


In [3]:
# dataframe for comparisons
df_comp = pd.read_csv(file_comp, sep='\t', header=0)
df_comp.head()

,ID,isohash
0,00063d88244921d6ec46aeab6866a8e2,fd5f9925c23132a6988d6b883cabf95d
1,00072cf107ae1349c8c59a15c5ce4af1,55060e6aa0f7a06dc70e163f2296eb95
2,00076733bdbce94d7e44eca84f1425f0,465f956423b4a18f2a53183fe4b5f682
3,000781b7a545fe723159e53127aff659,933f4c028a325d21969a109f37452e78
4,000a41cdca43be89ed62ea3abf2d0b64,fd393fdabc379273b867d42bc2be0ff4


HASH SUMMARY

In [6]:
hash_col = 'hash' if 'hash' in df.columns else 'isohash'

total = len(df)
na = df[hash_col].isna().sum()
unique = df[hash_col].nunique(dropna=True)
same = total - unique - na

print(f"TOTAL: {total}")
print(f"UNIQUE: {unique}")
print(f"NA: {na}")
print(f"SAME: {same}")

if 'max_iters' in df.columns:
    g = df.groupby('max_iters')[hash_col]
    s = pd.DataFrame({
        'TOTAL': g.size(),
        'NA': g.apply(lambda x: x.isna().sum()),
        'UNIQUE': g.nunique(dropna=True)
    })
    s['SAME'] = s['TOTAL'] - s['UNIQUE'] - s['NA']
    print("\nBY max_iters:")
    print(s.to_string())

TOTAL: 62918
UNIQUE: 49818
NA: 192
SAME: 12908

BY max_iters:
           TOTAL   NA  UNIQUE  SAME
max_iters                          
4          31459   51   26424  4984
12         31459  141   24956  6362


COMPARE MAX_ITERS WITH EACH OTHER

GET ALL DUPLICATE HASHES AND THEIR INSTANCES

In [7]:
# adding isohash to duplicated hashes of main to compare
dupe_rows = df[df['hash'].duplicated(keep=False)].sort_values('hash')
merged = dupe_rows.merge(
    df_comp[['ID','isohash']],
    on='ID',
    how='left'
)
print(merged.head(10))

                                 ID  max_iters              hash  \
0  0cb3538f0197f24fc57d79913b0531d9          4  0024b6da17ccc167   
1  0cb3538f0197f24fc57d79913b0531d9         12  0024b6da17ccc167   
2  4df590f124b66f482b7fea490be31ef5         12  00314387372ecd4e   
3  4df590f124b66f482b7fea490be31ef5          4  00314387372ecd4e   
4  5ea0f107a1905a8e0446af0c92d6a160         12  003d40de4b81afd1   
5  eea1f48a8ee1406432b1f6f64c989ddf         12  003d40de4b81afd1   
6  44bba88af37cce7712701583b09c3033          4  0047afd31e55affe   
7  b418ff1175e0b727ac09053fdec0f7a2          4  0047afd31e55affe   
8  f3e660ff5f79b4c8c465958d2a329c99          4  004a034b2cb92016   
9  f43ef6295cda449ce02a9707a7e20ecb          4  004a034b2cb92016   

                            isohash  
0  06e2c57d8aaf7b4856df6dacc8467baf  
1  06e2c57d8aaf7b4856df6dacc8467baf  
2  40223539dc253252f9645273943021af  
3  40223539dc253252f9645273943021af  
4  edbe2d20117eeffae3db6e6c3237e92b  
5  edbe2d20117eeffae3db

In [37]:
# adding hash of main to duplicated hashes of isohash
dupe_rows_comp = df_comp[df_comp['isohash'].duplicated(keep=False)].sort_values('isohash')
merged = dupe_rows_comp.merge(
    df[['ID','hash']],
    on='ID',
    how='left'
)
print(merged.head(10))

                                 ID                           isohash  \
0  4666906deb05406c0ecc18de81673d76  00a1a353e84e203de503f206c02b7a2c   
1  ae07d3502702cd744ad48e254abb192f  00a1a353e84e203de503f206c02b7a2c   
2  74acaa627ddc5fdab40d62172f56a827  00c5b384d9311d83da0be53e990b4904   
3  66ab346df72c5effa292342037bc3909  00c5b384d9311d83da0be53e990b4904   
4  df3da7a967605e804571a36c099416b7  00efc6678bd4710b52c3448cc6e9c5da   
5  847df6bb7bd0a8fcedb57e012cfe0014  00efc6678bd4710b52c3448cc6e9c5da   
6  56270bc03e3792c1c24d0deb72f75bbb  00efc6678bd4710b52c3448cc6e9c5da   
7  d93819b0aaf7569fe0b28e86db542330  00efc6678bd4710b52c3448cc6e9c5da   
8  ac0d8f93857783ae914af63f97bc33b3  00efc6678bd4710b52c3448cc6e9c5da   
9  f1c7fb2d19b357ee0ed4e9df28a1b533  012ae103f8674cab1b6a5721318dad86   

               hash  
0  d69d0ebf96ad5624  
1  919c77bbba16a4f3  
2  8660840629d5e0bb  
3  c46daf3059348902  
4  91fb463823e2a2d1  
5  6f6ee871820e226b  
6  b811a55547e77ae3  
7  efecea8e8d6e71d7 